# [busstop] 서울시 버스노선별 정류장별 시간대별 승하차 인원 → 정류장별 유동인구 평균 산출

### 기능
- 서울시 버스노선별 정류장별 시간대별 승하차 인원 현황 CSV를 읽는다.
- 2025년 6월~8월(202506, 202507, 202508) 데이터만 필터링한다.
- 각 행(노선-정류장-날짜)의 시간대별 승차총승객수 + 하차총승객수를 모두 합산하여 '총유동인구'를 계산한다.
- 정류장ID + 사용년월 기준 2단계 groupby로 **실제 존재하는 월 수** 기준 월평균 유동인구를 산출한다.
- 결과를 '버스정류장별_유동인구_평균_2025_06_08.csv'로 저장한다.

### 입력 파일
- `../data_processed/raw/bus_pop/서울시 버스노선별 정류장별 시간대별 승하차 인원 현황.csv`
- `../data_processed/raw/bus_pop/서울시버스정류소위치정보(20260108).xlsx`

### 출력 파일
- `../data_processed/processed/bus_pop/버스정류장별_유동인구_위치정보_평균_2025_06_08.csv`

In [1]:
import pandas as pd

In [2]:
# CSV 파일 읽기 (첫 번째 데이터 행이 영문 컬럼명이므로 skiprows로 제거)
df = pd.read_csv(
    r'../data_processed/raw/bus_pop/서울시 버스노선별 정류장별 시간대별 승하차 인원 현황.csv',
    encoding='cp949',
    low_memory=False,
    skiprows=[1],          # 0행=한글헤더(사용), 1행=영문헤더(제거)
)

# 사용년월을 문자열로 변환
df['사용년월'] = df['사용년월'].astype(str)

# 승차/하차 컬럼 추출 (원본 df 기준으로 탐색)
boarding_cols = [col for col in df.columns if '승차총승객수' in col]
alighting_cols = [col for col in df.columns if '하차총승객수' in col]
all_passenger_cols = boarding_cols + alighting_cols

print(f"전체 데이터: {len(df):,}행")
print(f"승차 컬럼 {len(boarding_cols)}개, 하차 컬럼 {len(alighting_cols)}개")

전체 데이터: 5,217,839행
승차 컬럼 24개, 하차 컬럼 24개


In [3]:
# 2025년 6월 ~ 8월 데이터 필터링 (.copy()로 독립 DataFrame 생성 → SettingWithCopyWarning 방지)
target_months = ['202506', '202507', '202508']
df_filtered = df.loc[df['사용년월'].isin(target_months)].copy()

print(f"필터링된 데이터: {len(df_filtered):,}행")

필터링된 데이터: 126,436행


In [4]:
# 승하차 컬럼 일괄 숫자 변환 (vectorized — apply 대신 DataFrame 단위 처리)
df_filtered.loc[:, all_passenger_cols] = (
    df_filtered[all_passenger_cols]
    .apply(pd.to_numeric, errors='coerce')
    .fillna(0)
)

# 각 행의 총 유동인구 계산 (승차 + 하차 전 시간대 합산)
df_filtered.loc[:, '총유동인구'] = df_filtered[all_passenger_cols].sum(axis=1)

In [5]:
# ── 2단계 groupby: 실제 존재하는 월 수 기준 평균 ──

# 1단계: 정류장ID + 사용년월별 유동인구 합계
monthly = (
    df_filtered
    .groupby(['표준버스정류장ID', '사용년월'], as_index=False)['총유동인구']
    .sum()
    .rename(columns={'총유동인구': '월별_유동인구'})
)

# 2단계: 정류장ID별 월평균 (실제 데이터가 존재하는 월 수로 나눔)
result = (
    monthly
    .groupby('표준버스정류장ID', as_index=False)['월별_유동인구']
    .mean()
    .rename(columns={'월별_유동인구': '유동인구_평균'})
)
result['유동인구_평균'] = result['유동인구_평균'].round(2)

In [6]:
# ID 타입 통일
result['표준버스정류장ID'] = result['표준버스정류장ID'].astype(str)

# ── 버스정류소 위치정보 매칭 ──
info = pd.read_excel(r"../data_processed/raw/bus_pop/서울시버스정류소위치정보(20260108).xlsx")
info.drop(columns=['정류소타입', 'ARS_ID'], inplace=True)
info.rename(columns={'X좌표': 'Longitude', 'Y좌표': 'Latitude'}, inplace=True)
info['NODE_ID'] = info['NODE_ID'].astype(str)

# 병합
result = (
    result
    .merge(info, how='left',
           left_on='표준버스정류장ID',
           right_on='NODE_ID')
    .drop(columns=['NODE_ID'])
)

print(f"위치정보 매칭 완료: {result['Longitude'].notna().sum():,}개 매칭 / {len(result):,}개 전체")


위치정보 매칭 완료: 10,652개 매칭 / 12,532개 전체


In [7]:
# CSV 파일로 저장 (유동인구 + 위치정보 포함)
output_path = '../data_processed/processed/bus_pop/버스정류장별_유동인구_위치정보_평균_2025_06_08.csv'
result.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"결과 저장: {output_path}")
print(f"총 {len(result):,}개 버스정류장")
print("\n결과 미리보기:")
result.head(10)

결과 저장: ../data_processed/processed/bus_pop/버스정류장별_유동인구_위치정보_평균_2025_06_08.csv
총 12,532개 버스정류장

결과 미리보기:


,표준버스정류장ID,유동인구_평균,정류소명,Longitude,Latitude
0,100000001,32746.00,종로2가사거리,126.987752,37.569806
1,100000002,136235.67,창경궁.서울대학교병원,126.996521,37.579433
2,100000003,198718.67,명륜3가.성대입구,126.998251,37.582580
3,100000004,54661.67,종로2가.삼일교,126.987613,37.568579
4,100000005,133375.33,혜화동로터리.여운형활동터,127.001744,37.586243
5,100000006,22132.00,경기상고,126.969363,37.587550
6,100000007,58665.67,신교동,126.970263,37.583296
7,100000008,19476.00,경기상고,126.969770,37.587191
8,100000009,21052.33,경복고교,126.970880,37.585160
9,100000010,7356.33,청운중학교,126.972674,37.587811


In [8]:
result.describe()

,유동인구_평균,Longitude,Latitude
count,12532.000000,10652.000000,10652.000000
mean,22781.987378,126.987280,37.550921
std,37038.120695,0.086306,0.055127
min,1.000000,126.797811,37.430520
25%,3529.085000,126.917576,37.502511
50%,9852.000000,126.996565,37.550659
75%,26244.252500,127.052399,37.591567
max,566519.670000,127.181734,37.690177
